# Reviewer-robust full analysis notebook

This notebook is intended to be the **single rerunnable revision analysis** for the CCM
review response. It reproduces the paper-facing results across all four datasets and both
outcomes, then adds the robustness checks requested by reviewers:

- S-learners: XGBoost, L1-regularized logistic S-learner, neural-network S-learner, and BART.
- R-learner/forest: `CausalForestDML` with cross-fitted nuisance models.
- Held-out 70/30 train/test split, stratified jointly by treatment and outcome.
- All learned preprocessing fit on training data only and applied unchanged to the test set.
- LR CATE-by-TTM interaction tests, CATE interval coverage, GATES, calibration, AUROC/Brier, and variable importance.
- Observational-cohort IPW versions of GATES and the CATE-by-TTM interaction diagnostic.
- Output CSVs/figures that can be pasted into `RESPONSE_TO_REVIEWERS.md`.

Run from `pooled/`:

```bash
jupyter nbconvert --to notebook --execute reviewerRobustAnalysis.ipynb \
  --output reviewerRobustAnalysis.executed.ipynb --ExecutePreprocessor.timeout=-1
```

The predictor CSVs are not committed to this repo. This notebook expects them at the paths in
`CSV` below, matching the existing dataset notebooks.


In [ ]:
# CONFIG
import os

def env_flag(name, default=True):
    raw = os.environ.get(name)
    if raw is None:
        return default
    return raw.strip().lower() not in {'0', 'false', 'no', 'off'}


SEED = 42
TEST_SIZE = 0.30
N_BOOT = 500
N_GATES_GROUPS = 5
PS_CLIP = (0.05, 0.95)
CORR_THRESHOLD = 0.70

# Runtime switches. Override on the cluster, e.g. RUN_BART_SLEARNER=0 for a smoke run.
RUN_XGB_SLEARNER = env_flag('RUN_XGB_SLEARNER', True)
RUN_LASSO_SLEARNER = env_flag('RUN_LASSO_SLEARNER', True)
RUN_NEURAL_SLEARNER = env_flag('RUN_NEURAL_SLEARNER', True)
RUN_BART_SLEARNER = env_flag('RUN_BART_SLEARNER', True)
RUN_CAUSAL_FOREST = env_flag('RUN_CAUSAL_FOREST', True)

# BART settings mirror the submitted notebooks. Increase draws for final archival runs if time allows.
BART_DRAW_SCALE = float(os.environ.get('BART_DRAW_SCALE', '1.0'))
BART_DRAWS = {
    'eICU': int(50 * BART_DRAW_SCALE),
    'PMAP': int(50 * BART_DRAW_SCALE),
    'MIMIC-IV': int(50 * BART_DRAW_SCALE),
    'HYPERION': int(100 * BART_DRAW_SCALE),
}
BART_TUNE = int(float(os.environ.get('BART_TUNE_SCALE', '1.0')) * 50)
BART_CHAINS = 2
BART_CORES = 1

REPO_ROOT = os.path.abspath('..')
OUTDIR = os.path.abspath('reviewer_robust_outputs')
os.makedirs(OUTDIR, exist_ok=True)

CSV = {
    'eICU':     os.path.join(REPO_ROOT, 'eICU',     'eICUPredictorsDiag.csv'),
    'PMAP':     os.path.join(REPO_ROOT, 'pmap',     'PMAP_Predictors4.csv'),
    'MIMIC-IV': os.path.join(REPO_ROOT, 'mimiciv',  'MIMIC_Predictors1.csv'),
    'HYPERION': os.path.join(REPO_ROOT, 'hyperion', 'predictorsDf.csv'),
}

DATASETS = [
    {'name': 'eICU',     'observational': True,  'treatment_col': 'TTM'},
    {'name': 'PMAP',     'observational': True,  'treatment_col': 'TTM'},
    {'name': 'MIMIC-IV', 'observational': True,  'treatment_col': 'TTM'},
    {'name': 'HYPERION', 'observational': False, 'treatment_col': 'TTM'},
]
OUTCOMES = {
    'mortality': 'mortality',
    'neuro': 'neuro_favorable',
}


In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import stats
from scipy.stats import chi2
import statsmodels.api as sm

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import KNNImputer
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.metrics import roc_auc_score, brier_score_loss
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from xgboost import XGBClassifier

try:
    from econml.dml import CausalForestDML
except Exception as e:
    CausalForestDML = None
    print('econml unavailable:', e)

try:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers
except Exception as e:
    tf = None
    keras = None
    layers = None
    print('TensorFlow unavailable:', e)

try:
    import pymc as pm
    import pymc_bart as pmb
except Exception as e:
    pm = None
    pmb = None
    print('PyMC/BART unavailable:', e)

np.random.seed(SEED)
pd.set_option('display.width', 180)
pd.set_option('display.max_columns', 80)


## Curated feature sets

These are copied from the current dataset DML/BART notebooks. They keep the analysis anchored
to clinically meaningful, pre-arrest/early-post-arrest variables and avoid training on the full
raw EHR feature space.


In [ ]:
CURATED = {
 'eICU': ['gender','age','bmi',
    'nurse_first_Non-Invasive BP Systolic','nurse_first_Non-Invasive BP Diastolic',
    'nurse_first_Non-Invasive BP Mean','nurse_first_Heart Rate','nurse_first_O2 Saturation',
    'lab_first_Respiratory Rate','lab_first_FiO2','nurse_first_GCS Total','nurse_first_Motor',
    'nurse_first_QTc','lab_first_pH','lab_first_paO2','lab_first_paCO2','lab_first_bicarbonate',
    'lab_first_lactate','lab_first_WBC x 1000','lab_first_Hgb','lab_first_platelets x 1000',
    'lab_first_sodium','lab_first_potassium','lab_first_BUN','lab_first_creatinine',
    'lab_first_calcium','lab_first_magnesium','lab_first_glucose','lab_first_troponin - T',
    'diagnosis_initial rhythm: ventricular fibrillation',
    'diagnosis_initial rhythm: ventricular tachycardia',
    'diagnosis_initial rhythm: pulseless electrical activity',
    'diagnosis_initial rhythm: asystole'],
 'PMAP': ['gender','age','first_mGCS','flo_first_r_cpn_glasgow_coma_scale_score',
    'flo_first_bp_systolic','flo_first_bp_diastolic','flo_first_r_map',
    'flo_first_r_ed_pre-arrival_pulse_(heart_rate)','flo_first_r_sao2','flo_first_r_fio2',
    'flo_first_r_sofa_score','flo_first_r_bmi','flo_first_r_pao2','flo_first_r_paco2',
    'flo_first_r_resp_ph','lab_first_lactate','lab_first_troponin','lab_first_hemoglobin',
    'lab_first_platelet_count','lab_first_creatinine,whole_blood','lab_first_glucose,whole_blood',
    'lab_first_potassium,whole_blood','lab_first_sodium,whole_blood','lab_first_calcium,_serum',
    'lab_first_magnesium','lab_first_jhm_ip_qtc_ainterval_(sec)','asystole','pea','VF'],
 'MIMIC-IV': ['gender','age','bmi','first_mGCS',
    'chart_first_heart_rate','chart_first_o2_saturation_pulseoxymetry','chart_first_respiratory_rate',
    'chart_first_fio2_(ch)','chart_first_non_invasive_blood_pressure_systolic',
    'chart_first_non_invasive_blood_pressure_diastolic','chart_first_non_invasive_blood_pressure_mean',
    'chart_first_ph_(arterial)','chart_first_arterial_o2_pressure','chart_first_arterial_co2_pressure',
    'chart_first_lactic_acid','chart_first_wbc','chart_first_hemoglobin','lab_first_platelet_count',
    'chart_first_sodium_(serum)','lab_first_potassium_(serum)','chart_first_bun',
    'chart_first_creatinine_(serum)','chart_first_calcium_non-ionized','chart_first_magnesium',
    'chart_first_glucose_(serum)','lab_first_troponin-t','chart_first_qtc',
    'long_title_ventricular_fibrillation'],
 'HYPERION': ['J0_AGE','J0_SEX','J0_BMI','J0_PAS','J0_PAD','J0_PAM','J0_FC','J0_SPO2',
    'J0_GLASGOW','J0_MOTRICE','J0_RYTHM','J0_NOFLOW','J0_LOWFLOW','J0_IGSII',
    'BIO_LEUCO','BIO_HEMO','BIO_PLAQ','BIO_SODIUM','BIO_POTAS','BIO_UREE','BIO_CREAT',
    'BIO_CALCIUM','BIO_MAGNE','BIO_GLYCEMI','BIO_LACTAT','BIO_TROPO','BIO_PH','BIO_PAO2',
    'BIO_PACO2','BIO_BICARB'],
}


## Dataset loaders

Loaders replicate the current cohort filters and canonicalize treatment/outcome columns to:
`TTM`, `mortality`, and `neuro_favorable`.


In [ ]:
UNSCORABLE = 'Unable to score due to medication'


def require_file(path):
    if not os.path.exists(path):
        raise FileNotFoundError(
            f'Missing {path}. Restore the predictor CSVs before executing the notebook.'
        )


def load_eicu():
    require_file(CSV['eICU'])
    df = pd.read_csv(CSV['eICU'])
    f = (df['LastMGCS'] != UNSCORABLE) & (~df['LastMGCS'].isna())
    f = f & (df['FirstMGCSTime'] != df['LastMGCSTime'])
    for c in ['FirstGCS', 'FirstMGCS', 'LastMGCS', 'LastGCS']:
        if c in df.columns:
            df.loc[df[c] == UNSCORABLE, c] = np.nan
    df.loc[df['DeathAtDischarge'] == 1, 'LastMGCS'] = 1
    df['gender'] = (df['gender'] == 'Male').astype(int)
    df.loc[f, 'LastMGCSPositive'] = (df.loc[f, 'LastMGCS'].astype(float) == 6).astype(int)
    df = df[f & (df['nurse_first_Motor'] != 6) & ~df['Hypothermia'].isna()].copy()
    return df.rename(columns={
        'Hypothermia': 'TTM',
        'DeathAtDischarge': 'mortality',
        'LastMGCSPositive': 'neuro_favorable',
    })


def _load_epic_style(name):
    require_file(CSV[name])
    df = pd.read_csv(CSV[name])
    f = (df['first_mGCS_time'] != df['last_mGCS_time'])
    df.loc[df['death_at_disch'] == 1, 'last_mGCS'] = 1
    df.loc[f, 'LastMGCSPositive'] = (df.loc[f, 'last_mGCS'].astype(float) == 6).astype(int)
    df = df[(df['first_mGCS'] != 6) & ~df['hypothermia'].isna()].copy()
    return df.rename(columns={
        'hypothermia': 'TTM',
        'death_at_disch': 'mortality',
        'LastMGCSPositive': 'neuro_favorable',
    })


def load_pmap():
    return _load_epic_style('PMAP')


def load_mimic():
    return _load_epic_style('MIMIC-IV')


def load_hyperion():
    require_file(CSV['HYPERION'])
    df = pd.read_csv(CSV['HYPERION'])
    df = df[df['groupe'] != 2].copy()
    df['TTM'] = (df['groupe'] == 1).astype(int)
    return df.rename(columns={
        'hospital_mortality': 'mortality',
        'CPC12': 'neuro_favorable',
    })


LOADERS = {
    'eICU': load_eicu,
    'PMAP': load_pmap,
    'MIMIC-IV': load_mimic,
    'HYPERION': load_hyperion,
}


def load_analysis_frame(dataset, outcome):
    df = LOADERS[dataset]()
    cols = [c for c in CURATED[dataset] if c in df.columns]
    missing = [c for c in CURATED[dataset] if c not in df.columns]
    X = df[cols].apply(pd.to_numeric, errors='coerce')
    T = pd.to_numeric(df['TTM'], errors='coerce')
    y = pd.to_numeric(df[outcome], errors='coerce')
    keep = T.notna() & y.notna()
    out = X.loc[keep].reset_index(drop=True)
    out['TTM'] = T.loc[keep].astype(int).reset_index(drop=True)
    out[outcome] = y.loc[keep].astype(int).reset_index(drop=True)
    meta = {
        'dataset': dataset,
        'outcome': outcome,
        'n': int(keep.sum()),
        'n_ttm': int(T.loc[keep].sum()),
        'n_features': len(cols),
        'missing_curated_features': missing,
    }
    return out, cols, meta


cohort_rows = []
for ds in DATASETS:
    for outcome_name, outcome_col in OUTCOMES.items():
        try:
            _, _, meta = load_analysis_frame(ds['name'], outcome_col)
            meta['outcome_name'] = outcome_name
            cohort_rows.append(meta)
        except Exception as e:
            cohort_rows.append({
                'dataset': ds['name'],
                'outcome': outcome_col,
                'outcome_name': outcome_name,
                'status': f'missing: {e}',
            })

cohort_table = pd.DataFrame(cohort_rows)
cohort_table.to_csv(os.path.join(OUTDIR, 'cohort_table.csv'), index=False)
display(cohort_table)


## Preprocessing and common metrics

The key reviewer-facing point: all learned preprocessing is fit on training data only.


In [ ]:
def make_stratify(y, T):
    return pd.Series(y).astype(str) + '_' + pd.Series(T).astype(str)


def split_frame(df, feature_cols, outcome_col):
    strat = make_stratify(df[outcome_col], df['TTM'])
    train_idx, test_idx = train_test_split(
        np.arange(len(df)),
        test_size=TEST_SIZE,
        random_state=SEED,
        stratify=strat,
    )
    return train_idx, test_idx


def fit_preprocessor(X_train):
    numeric_cols = [c for c in X_train.columns if X_train[c].dropna().nunique() > 2]
    binary_cols = [c for c in X_train.columns if c not in numeric_cols]
    transformers = []
    if numeric_cols:
        transformers.append((
            'num',
            Pipeline([
                ('scale', StandardScaler()),
                ('impute', KNNImputer(n_neighbors=10, keep_empty_features=True)),
            ]),
            numeric_cols,
        ))
    if binary_cols:
        transformers.append((
            'bin',
            Pipeline([
                ('impute', KNNImputer(n_neighbors=10, keep_empty_features=True)),
            ]),
            binary_cols,
        ))
    return ColumnTransformer(transformers=transformers, remainder='drop', verbose_feature_names_out=False)


def transform_split(df, feature_cols, train_idx, test_idx, outcome_col):
    X_train_raw = df.loc[train_idx, feature_cols]
    X_test_raw = df.loc[test_idx, feature_cols]
    prep = fit_preprocessor(X_train_raw)
    X_train = pd.DataFrame(prep.fit_transform(X_train_raw), columns=prep.get_feature_names_out())
    X_test = pd.DataFrame(prep.transform(X_test_raw), columns=prep.get_feature_names_out())
    y_train = df.loc[train_idx, outcome_col].astype(int).to_numpy()
    y_test = df.loc[test_idx, outcome_col].astype(int).to_numpy()
    T_train = df.loc[train_idx, 'TTM'].astype(int).to_numpy()
    T_test = df.loc[test_idx, 'TTM'].astype(int).to_numpy()
    return X_train, X_test, y_train, y_test, T_train, T_test, prep


def bootstrap_auc_ci(y, p, n_boot=N_BOOT, seed=SEED):
    rng = np.random.default_rng(seed)
    aucs = []
    y = np.asarray(y)
    p = np.asarray(p)
    for _ in range(n_boot):
        idx = rng.integers(0, len(y), len(y))
        if len(np.unique(y[idx])) < 2:
            continue
        aucs.append(roc_auc_score(y[idx], p[idx]))
    if not aucs:
        return np.nan, np.nan
    return tuple(np.percentile(aucs, [2.5, 97.5]))


def calibration_slope(y, p):
    p = np.clip(np.asarray(p), 1e-6, 1 - 1e-6)
    logit_p = np.log(p / (1 - p))
    X = sm.add_constant(logit_p)
    try:
        m = sm.Logit(y, X).fit(disp=False)
        return float(m.params[1])
    except Exception:
        return np.nan


def lrt_cate_interaction(y, T, cate):
    cate = np.asarray(cate, float)
    if np.ptp(cate) < 1e-12:
        return {'lr_stat': np.nan, 'p': np.nan, 'note': 'CATE constant'}
    d = pd.DataFrame({'const': 1.0, 'T': np.asarray(T, float), 'cate': cate})
    d['tx'] = d['T'] * d['cate']
    y = np.asarray(y, float)
    try:
        m0 = sm.Logit(y, d[['const', 'T', 'cate']]).fit(disp=False)
        m1 = sm.Logit(y, d[['const', 'T', 'cate', 'tx']]).fit(disp=False)
        lr = 2 * (m1.llf - m0.llf)
        return {'lr_stat': float(lr), 'p': float(chi2.sf(lr, 1)), 'note': ''}
    except Exception as e:
        return {'lr_stat': np.nan, 'p': np.nan, 'note': str(e)}


def ipw_interaction_test(y, T, cate, ps, clip=PS_CLIP):
    T = np.asarray(T, float)
    y = np.asarray(y, float)
    cate = np.asarray(cate, float)
    ps = np.clip(np.asarray(ps, float), *clip)
    if np.ptp(cate) < 1e-12:
        return {'wald_p': np.nan, 'note': 'CATE constant'}
    w = np.where(T == 1, 1.0 / ps, 1.0 / (1.0 - ps))
    d = pd.DataFrame({'const': 1.0, 'T': T, 'cate': cate})
    d['tx'] = d['T'] * d['cate']
    try:
        m = sm.GLM(y, d[['const', 'T', 'cate', 'tx']], family=sm.families.Binomial(),
                   freq_weights=w).fit(cov_type='HC1')
        return {'wald_p': float(m.pvalues['tx']), 'note': ''}
    except Exception as e:
        return {'wald_p': np.nan, 'note': str(e)}


def run_gates(cate, y, T, ps=None, observational=False, title='', save_prefix=None):
    d = pd.DataFrame({'y': np.asarray(y, float), 'T': np.asarray(T, float), 'cate': np.asarray(cate, float)})
    if observational and ps is not None:
        ps = np.clip(np.asarray(ps, float), *PS_CLIP)
        d['w'] = np.where(d['T'] == 1, 1.0 / ps, 1.0 / (1.0 - ps))
    else:
        d['w'] = 1.0
    d['group'] = pd.qcut(d['cate'].rank(method='first'), q=N_GATES_GROUPS, labels=False) + 1
    rows = []
    for g, sub in d.groupby('group'):
        m = sm.WLS(sub['y'], sm.add_constant(sub['T']), weights=sub['w']).fit()
        ci = np.asarray(m.conf_int())
        rows.append({
            'group': int(g),
            'n': len(sub),
            'mean_cate': sub['cate'].mean(),
            'gate': m.params.iloc[1],
            'ci_low': ci[1, 0],
            'ci_high': ci[1, 1],
        })
    gdf = pd.DataFrame(rows)
    gd = pd.get_dummies(d['group'].astype(int), prefix='g', drop_first=True).astype(float)
    Xr = sm.add_constant(pd.concat([d[['T']], gd], axis=1))
    Xf = Xr.copy()
    for c in gd.columns:
        Xf[f'T_x_{c}'] = d['T'].values * gd[c].values
    mf = sm.WLS(d['y'], Xf, weights=d['w']).fit()
    mr = sm.WLS(d['y'], Xr, weights=d['w']).fit()
    f_stat, f_p, _ = mf.compare_f_test(mr)
    rho, rho_p = stats.spearmanr(gdf['group'], gdf['gate'])
    if save_prefix:
        fig, ax = plt.subplots(figsize=(5.8, 4.0))
        ax.errorbar(gdf['group'], gdf['gate'],
                    yerr=[gdf['gate'] - gdf['ci_low'], gdf['ci_high'] - gdf['gate']],
                    fmt='o-', capsize=4)
        ax.axhline(0, color='0.4', ls='--', lw=1)
        ax.set_xlabel('Predicted CATE quintile')
        ax.set_ylabel('Group average treatment effect')
        ax.set_title(f'{title}\nF p={f_p:.3f}; rho={rho:.2f} (p={rho_p:.3f})')
        fig.tight_layout()
        fig.savefig(os.path.join(OUTDIR, f'{save_prefix}_gates.png'), dpi=180)
        plt.close(fig)
    return gdf, {'f_stat': float(f_stat), 'f_p': float(f_p), 'spearman_rho': float(rho), 'spearman_p': float(rho_p)}


## Model definitions

In [ ]:
def slearner_design(X, T):
    Xd = X.copy()
    Xd['TTM'] = np.asarray(T, float)
    return Xd


def predict_slearner_cate(model, X):
    x0 = X.copy()
    x1 = X.copy()
    x0['TTM'] = 0.0
    x1['TTM'] = 1.0
    p0 = model.predict_proba(x0)[:, 1]
    p1 = model.predict_proba(x1)[:, 1]
    return p1 - p0, p0, p1


def fit_xgb_slearner(X_train, T_train, y_train):
    model = XGBClassifier(eval_metric='logloss', random_state=SEED, n_jobs=-1)
    grid = {
        'n_estimators': [25, 50, 100, 200],
        'max_depth': [2, 5, 10],
        'learning_rate': [0.03, 0.1],
    }
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    gs = GridSearchCV(model, grid, cv=cv, scoring='roc_auc', n_jobs=-1, verbose=0)
    gs.fit(slearner_design(X_train, T_train), y_train)
    return gs.best_estimator_, gs.best_params_


def fit_lasso_slearner(X_train, T_train, y_train):
    Xd = slearner_design(X_train, T_train)
    base_cols = list(X_train.columns)
    for c in base_cols:
        Xd[f'{c}_x_TTM'] = Xd[c] * Xd['TTM']
    model = LogisticRegressionCV(
        penalty='l1',
        solver='saga',
        Cs=[1e-3, 1e-2, 1e-1, 1.0],
        cv=5,
        scoring='neg_log_loss',
        max_iter=20000,
        n_jobs=-1,
        random_state=SEED,
    )
    model.fit(Xd, y_train)
    return model, {'chosen_C': float(model.C_[0]), 'interaction_cols': [f'{c}_x_TTM' for c in base_cols]}


def predict_lasso_slearner(model, X, interaction_cols):
    def build(T_value):
        Xd = X.copy()
        Xd['TTM'] = float(T_value)
        for c in X.columns:
            Xd[f'{c}_x_TTM'] = Xd[c] * Xd['TTM']
        return Xd
    p0 = model.predict_proba(build(0))[:, 1]
    p1 = model.predict_proba(build(1))[:, 1]
    return p1 - p0, p0, p1


def fit_neural_slearner(X_train, T_train, y_train, dataset):
    if keras is None:
        raise RuntimeError('TensorFlow/Keras is not available')
    tf.keras.utils.set_random_seed(SEED)
    Xd = slearner_design(X_train, T_train).to_numpy(dtype='float32')
    y = np.asarray(y_train).astype('float32')
    nn_cfg = {
        'eICU':     {'layers': [64, 64, 32],       'dropout': 0.5, 'lr': 1e-3, 'epochs': 20},
        'PMAP':     {'layers': [64, 64, 64, 64, 32],'dropout': 0.2, 'lr': 1e-4, 'epochs': 50},
        'MIMIC-IV': {'layers': [64, 64, 64, 32],   'dropout': 0.2, 'lr': 1e-3, 'epochs': 60},
        'HYPERION': {'layers': [64],               'dropout': 0.5, 'lr': 1e-3, 'epochs': 15},
    }[dataset]
    inp = keras.Input(shape=(Xd.shape[1],))
    z = inp
    for width in nn_cfg['layers']:
        z = layers.Dense(width, activation='relu')(z)
        z = layers.Dropout(nn_cfg['dropout'])(z)
    out = layers.Dense(1, activation='sigmoid')(z)
    model = keras.Model(inp, out)
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=nn_cfg['lr']),
                  loss='binary_crossentropy', metrics=[keras.metrics.AUC(name='auc')])
    model.fit(Xd, y, batch_size=32, epochs=nn_cfg['epochs'], validation_split=0.2, verbose=0)
    return model, nn_cfg


def predict_neural_slearner(model, X):
    def build(T_value):
        Xd = X.copy()
        Xd['TTM'] = float(T_value)
        return Xd.to_numpy(dtype='float32')
    p0 = model.predict(build(0), verbose=0).ravel()
    p1 = model.predict(build(1), verbose=0).ravel()
    return p1 - p0, p0, p1


def fit_bart_slearner(X_train, T_train, y_train, dataset):
    if pm is None or pmb is None:
        raise RuntimeError('PyMC/pymc_bart is not available')
    Xd = slearner_design(X_train, T_train).to_numpy(dtype='float64')
    y = np.asarray(y_train).astype(int)
    m_trees = {'eICU': 30, 'PMAP': 30, 'MIMIC-IV': 50, 'HYPERION': 30}[dataset]
    with pm.Model() as model:
        X_shared = pm.Data('X_shared', Xd)
        u = pmb.BART('u', X=X_shared, Y=y, m=m_trees)
        p = pm.Deterministic('p', pm.math.sigmoid(u))
        pm.Bernoulli('y', p=p, observed=y)
        trace = pm.sample(
            draws=BART_DRAWS[dataset],
            tune=BART_TUNE,
            chains=BART_CHAINS,
            cores=BART_CORES,
            random_seed=SEED,
            progressbar=True,
        )
    return model, trace, {'m_trees': m_trees, 'draws': BART_DRAWS[dataset], 'tune': BART_TUNE}


def predict_bart_slearner(model, trace, X):
    def build(T_value):
        Xd = X.copy()
        Xd['TTM'] = float(T_value)
        return Xd.to_numpy(dtype='float64')
    preds = []
    with model:
        for val in [0, 1]:
            pm.set_data({'X_shared': build(val)})
            ppc = pm.sample_posterior_predictive(trace, var_names=['p'], random_seed=SEED, progressbar=False)
            arr = np.asarray(ppc.posterior_predictive['p'])
            preds.append(arr.reshape(-1, arr.shape[-1]).mean(axis=0))
    p0, p1 = preds
    return p1 - p0, p0, p1


def fit_causal_forest(X_train, T_train, y_train, dataset):
    if CausalForestDML is None:
        raise RuntimeError('econml is not available')
    # Keep HYPERION settings aligned with the submitted DML notebook; observational settings
    # match the current revision sensitivity notebooks.
    if dataset == 'HYPERION':
        model_y = XGBClassifier(max_depth=25, n_estimators=100, random_state=SEED, eval_metric='logloss')
        model_t = XGBClassifier(max_depth=10, n_estimators=20, random_state=SEED, eval_metric='logloss')
    elif dataset == 'PMAP':
        model_y = XGBClassifier(max_depth=5, n_estimators=50, random_state=SEED, eval_metric='logloss')
        model_t = XGBClassifier(max_depth=2, n_estimators=20, random_state=SEED, eval_metric='logloss')
    else:
        model_y = XGBClassifier(max_depth=3, n_estimators=50, random_state=SEED, eval_metric='logloss')
        model_t = XGBClassifier(max_depth=2, n_estimators=20, random_state=SEED, eval_metric='logloss')
    cf = CausalForestDML(
        model_y=model_y,
        model_t=model_t,
        discrete_treatment=True,
        discrete_outcome=True,
        random_state=SEED,
        n_jobs=-1,
    )
    cf.fit(y_train, T_train, X=X_train, cache_values=True)
    return cf, {'model_y': str(model_y), 'model_t': str(model_t)}


def cf_propensity(cf, X):
    preds = []
    for mc in cf.models_t:
        for mdl in mc:
            p = mdl.predict_proba(np.asarray(X))
            preds.append(p[:, 1] if p.ndim == 2 else np.ravel(p))
    return np.clip(np.mean(np.vstack(preds), axis=0), 1e-6, 1 - 1e-6)


## Run all datasets, outcomes, and models

In [ ]:
RESULTS = []
GATES_ROWS = []
IMPORTANCE_ROWS = []


def add_result(row):
    RESULTS.append(row)
    pd.DataFrame(RESULTS).to_csv(os.path.join(OUTDIR, 'all_model_results_partial.csv'), index=False)


def evaluate_slearner(dataset, outcome_name, outcome_col, observational, model_name, cate, p_obs, p0, p1, y_test, T_test, extra=None):
    lrt = lrt_cate_interaction(y_test, T_test, cate)
    auc = roc_auc_score(y_test, p_obs) if len(np.unique(y_test)) == 2 else np.nan
    ci_low, ci_high = bootstrap_auc_ci(y_test, p_obs)
    row = {
        'dataset': dataset,
        'outcome': outcome_name,
        'model': model_name,
        'n_test': len(y_test),
        'lr_p': lrt['p'],
        'lr_note': lrt['note'],
        'auroc': auc,
        'auroc_ci_low': ci_low,
        'auroc_ci_high': ci_high,
        'brier': brier_score_loss(y_test, p_obs),
        'calibration_slope': calibration_slope(y_test, p_obs),
        'mean_cate': float(np.mean(cate)),
        'cate_p05': float(np.percentile(cate, 5)),
        'cate_p50': float(np.percentile(cate, 50)),
        'cate_p95': float(np.percentile(cate, 95)),
    }
    if extra:
        row.update(extra)
    add_result(row)


def run_dataset_outcome(dataset, outcome_name, outcome_col, observational):
    print(f'\n===== {dataset} / {outcome_name} =====')
    try:
        df, feature_cols, meta = load_analysis_frame(dataset, outcome_col)
    except Exception as e:
        add_result({
            'dataset': dataset,
            'outcome': outcome_name,
            'model': 'all',
            'status': f'skipped before modeling: {e}',
        })
        print('Skipped:', e)
        return
    train_idx, test_idx = split_frame(df, feature_cols, outcome_col)
    X_train, X_test, y_train, y_test, T_train, T_test, prep = transform_split(
        df, feature_cols, train_idx, test_idx, outcome_col
    )
    print(f'n={len(df)}; train={len(train_idx)}; test={len(test_idx)}; features={len(feature_cols)}')

    if RUN_XGB_SLEARNER:
        try:
            model, params = fit_xgb_slearner(X_train, T_train, y_train)
            cate, p0, p1 = predict_slearner_cate(model, X_test)
            p_obs = np.where(T_test == 1, p1, p0)
            evaluate_slearner(dataset, outcome_name, outcome_col, observational, 'S-learner XGBoost',
                              cate, p_obs, p0, p1, y_test, T_test, params)
            if hasattr(model, 'feature_importances_'):
                for col, val in zip(slearner_design(X_train, T_train).columns, model.feature_importances_):
                    IMPORTANCE_ROWS.append({'dataset': dataset, 'outcome': outcome_name, 'model': 'S-learner XGBoost',
                                            'feature': col, 'importance': float(val)})
        except Exception as e:
            add_result({'dataset': dataset, 'outcome': outcome_name, 'model': 'S-learner XGBoost', 'status': f'failed: {e}'})
            print('XGBoost failed:', e)

    if RUN_LASSO_SLEARNER:
        try:
            model, info = fit_lasso_slearner(X_train, T_train, y_train)
            cate, p0, p1 = predict_lasso_slearner(model, X_test, info['interaction_cols'])
            p_obs = np.where(T_test == 1, p1, p0)
            coefs = pd.Series(model.coef_[0], index=slearner_design(X_train, T_train).columns.tolist() + info['interaction_cols'])
            n_nonzero_interactions = int((coefs[info['interaction_cols']] != 0).sum())
            evaluate_slearner(dataset, outcome_name, outcome_col, observational, 'S-learner L1 logistic',
                              cate, p_obs, p0, p1, y_test, T_test,
                              {'chosen_C': info['chosen_C'], 'nonzero_interactions': n_nonzero_interactions,
                               'n_interactions': len(info['interaction_cols'])})
        except Exception as e:
            add_result({'dataset': dataset, 'outcome': outcome_name, 'model': 'S-learner L1 logistic', 'status': f'failed: {e}'})
            print('L1 logistic failed:', e)

    if RUN_NEURAL_SLEARNER:
        try:
            model, cfg = fit_neural_slearner(X_train, T_train, y_train, dataset)
            cate, p0, p1 = predict_neural_slearner(model, X_test)
            p_obs = np.where(T_test == 1, p1, p0)
            evaluate_slearner(dataset, outcome_name, outcome_col, observational, 'S-learner neural network',
                              cate, p_obs, p0, p1, y_test, T_test, cfg)
        except Exception as e:
            add_result({'dataset': dataset, 'outcome': outcome_name, 'model': 'S-learner neural network', 'status': f'failed: {e}'})
            print('Neural failed:', e)

    if RUN_BART_SLEARNER:
        try:
            model, trace, cfg = fit_bart_slearner(X_train, T_train, y_train, dataset)
            cate, p0, p1 = predict_bart_slearner(model, trace, X_test)
            p_obs = np.where(T_test == 1, p1, p0)
            evaluate_slearner(dataset, outcome_name, outcome_col, observational, 'S-learner BART',
                              cate, p_obs, p0, p1, y_test, T_test, cfg)
        except Exception as e:
            add_result({'dataset': dataset, 'outcome': outcome_name, 'model': 'S-learner BART', 'status': f'failed: {e}'})
            print('BART failed:', e)

    if RUN_CAUSAL_FOREST:
        try:
            cf, cfg = fit_causal_forest(X_train, T_train, y_train, dataset)
            cate = np.ravel(cf.effect(X_test))
            lo, hi = cf.effect_interval(X_test, alpha=0.05)
            lo = np.ravel(lo)
            hi = np.ravel(hi)
            lrt = lrt_cate_interaction(y_test, T_test, cate)
            ps = cf_propensity(cf, X_test) if observational else np.repeat(T_train.mean(), len(T_test))
            ipw = ipw_interaction_test(y_test, T_test, cate, ps) if observational else {'wald_p': np.nan, 'note': 'randomized'}
            gates, gates_stats = run_gates(
                cate, y_test, T_test, ps=ps, observational=observational,
                title=f'{dataset} {outcome_name} CausalForestDML',
                save_prefix=f'{dataset}_{outcome_name}_causal_forest'.replace(' ', '_').replace('/', '_')
            )
            gates.insert(0, 'dataset', dataset)
            gates.insert(1, 'outcome', outcome_name)
            gates.insert(2, 'model', 'CausalForestDML/R-learner')
            GATES_ROWS.extend(gates.to_dict('records'))
            row = {
                'dataset': dataset,
                'outcome': outcome_name,
                'model': 'CausalForestDML/R-learner',
                'n_test': len(y_test),
                'lr_p': lrt['p'],
                'lr_note': lrt['note'],
                'ipw_interaction_p': ipw['wald_p'],
                'gates_f_p': gates_stats['f_p'],
                'gates_spearman_rho': gates_stats['spearman_rho'],
                'gates_spearman_p': gates_stats['spearman_p'],
                'pct_cate_ci_cross_zero': float(np.mean((lo < 0) & (hi > 0))),
                'mean_cate': float(np.mean(cate)),
                'cate_p05': float(np.percentile(cate, 5)),
                'cate_p50': float(np.percentile(cate, 50)),
                'cate_p95': float(np.percentile(cate, 95)),
            }
            row.update(cfg)
            add_result(row)
            if hasattr(cf, 'feature_importances_'):
                for col, val in zip(X_train.columns, cf.feature_importances_):
                    IMPORTANCE_ROWS.append({'dataset': dataset, 'outcome': outcome_name,
                                            'model': 'CausalForestDML/R-learner',
                                            'feature': col, 'importance': float(val)})
        except Exception as e:
            add_result({'dataset': dataset, 'outcome': outcome_name, 'model': 'CausalForestDML/R-learner', 'status': f'failed: {e}'})
            print('CausalForest failed:', e)


for ds in DATASETS:
    for outcome_name, outcome_col in OUTCOMES.items():
        run_dataset_outcome(ds['name'], outcome_name, outcome_col, ds['observational'])

results = pd.DataFrame(RESULTS)
gates_table = pd.DataFrame(GATES_ROWS)
importance_table = pd.DataFrame(IMPORTANCE_ROWS)

results.to_csv(os.path.join(OUTDIR, 'all_model_results.csv'), index=False)
gates_table.to_csv(os.path.join(OUTDIR, 'gates_results.csv'), index=False)
importance_table.to_csv(os.path.join(OUTDIR, 'variable_importance.csv'), index=False)

display(results)


## Paper-facing summary tables

In [ ]:
results = pd.DataFrame(RESULTS)

for col in ['lr_p', 'auroc', 'auroc_ci_low', 'auroc_ci_high', 'brier',
            'calibration_slope', 'nonzero_interactions', 'n_interactions', 'chosen_C']:
    if col not in results.columns:
        results[col] = np.nan

table2 = results.pivot_table(
    index=['dataset', 'outcome', 'model'],
    values=['lr_p', 'auroc', 'auroc_ci_low', 'auroc_ci_high', 'brier', 'calibration_slope'],
    aggfunc='first',
).reset_index()
table2.to_csv(os.path.join(OUTDIR, 'table2_model_performance_and_lr.csv'), index=False)
display(table2.round(4))

cf_summary = results[results['model'].eq('CausalForestDML/R-learner')].copy()
cf_summary.to_csv(os.path.join(OUTDIR, 'causal_forest_hte_summary.csv'), index=False)
display(cf_summary.round(4))

lasso_summary = results[results['model'].eq('S-learner L1 logistic')][
    ['dataset', 'outcome', 'nonzero_interactions', 'n_interactions', 'chosen_C', 'lr_p']
].copy()
lasso_summary.to_csv(os.path.join(OUTDIR, 'lasso_slearner_summary.csv'), index=False)
display(lasso_summary)


## Response-letter text snippets

These lines are deliberately terse so they can be pasted into `RESPONSE_TO_REVIEWERS.md`.


In [ ]:
def fmt_p(x):
    if pd.isna(x):
        return 'NA'
    return '<0.001' if x < 0.001 else f'{x:.3f}'


print('--- Train/test and preprocessing ---')
print(f'Within each dataset, patients were randomly split {int((1-TEST_SIZE)*100)}/{int(TEST_SIZE*100)} into training and held-out test sets, stratified jointly by treatment and outcome. Standard scaling, KNN imputation (k=10), and all model tuning were fit on training data only.')

print('\n--- CausalForestDML/R-learner HTE summary ---')
for _, r in cf_summary.iterrows():
    print(f"{r['dataset']} {r['outcome']}: LR p={fmt_p(r['lr_p'])}; "
          f"GATES F p={fmt_p(r['gates_f_p'])}; "
          f"CATE 95% CIs crossing zero={r['pct_cate_ci_cross_zero']:.1%}; "
          f"IPW interaction p={fmt_p(r.get('ipw_interaction_p', np.nan))}.")

print('\n--- L1 S-learner regularization summary ---')
for _, r in lasso_summary.iterrows():
    print(f"{r['dataset']} {r['outcome']}: {int(r['nonzero_interactions'])}/{int(r['n_interactions'])} treatment interactions survived the L1 penalty; LR p={fmt_p(r['lr_p'])}.")


## Reviewer response strategy

Use the notebook outputs to fill the response document, but keep the narrative conservative:

1. The primary result remains the per-dataset held-out analysis. The pooled observational analysis is a sensitivity analysis because exposure ascertainment and case mix differ.
2. For observational data, explain confounding control in layers: covariate-conditioned S-learners, orthogonalized CausalForestDML nuisance models, propensity overlap diagnostics, and IPW GATES/interaction diagnostics.
3. For dimensionality concerns, emphasize curated HYPERION-anchored covariates, rare-feature and treatment-collinearity filters, training-only feature/preprocessing steps, L1 S-learner sensitivity, and the harmonized pooled analysis.
4. Be explicit that small effects in small subgroups remain underpowered; this is a limitation, not a failure of the analysis.
